# [Pipeline Title]

---

## 1. Problem Framing

### Business Problem

<!-- TODO: Describe the business problem in plain language. Who is affected?
What decision is currently made without data? What does this pipeline enable? -->

This pipeline answers the question: **[TODO: one sentence business question]**

The deployed output is a **[TODO: score / flag / recommendation]** visible on
[TODO: where in the application].

### Who Cares About This

- **[Role]** — [why they care]
- **[Role]** — [why they care]

### Predictive vs. Explanatory

This pipeline uses a **[predictive / explanatory]** approach.

<!-- TODO: Justify the choice. Predictive = reliable scores on new data.
Explanatory = understanding and quantifying relationships.
If predictive: focus is out-of-sample performance, interpretability is secondary.
If explanatory: coefficients and defensibility matter more than accuracy. -->

### Success Metrics

- **Primary:** [metric] — [one sentence justification]
- **Secondary:** [metrics]
- **Operational threshold:** [TODO: what error is worse and why]

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Resolve paths relative to the ML_Pipelines/ project root
sys.path.append(os.path.dirname(os.path.abspath('.')))
os.chdir('..')

# TODO: Import the correct prepare function for this pipeline
# from functions.fn_eda_custom import prepare_residents   # residents domain
# from functions.fn_eda_custom import prepare_donors       # donors domain
# from functions.fn_eda_custom import prepare_social_media # social media domain

from functions.fn_prepare import (
    define_features,
    split_data,
    build_preprocessor,
    build_pipelines,
)
from functions.fn_model_predict import (
    run_cross_validation,
    tune_model,
    evaluate_final_model,
    save_model,
)

# TODO: For explanatory pipelines only — remove for predictive
# from functions.fn_model_causal import fit_causal_classification, get_coefficients
# from functions.fn_model_causal import fit_causal_regression, get_coefficients

print("All imports successful.")

### 2.1 Load and Prepare Data

`prepare_[domain]()` encodes every cleaning and feature engineering decision from
`eda_[domain].ipynb` into a single reproducible call. No manual steps, no intermediate
files — running this cell always reflects the current state of those decisions.

**Tables joined:** `[TODO: list tables]`

**Key preparation decisions encoded:**
- [TODO: list the major decisions from your EDA — copy from the EDA summary cell]
- Structural columns dropped: [list]
- [Feature engineering decisions]
- `[target]` engineered from `[source column / logic]`

In [ ]:
# TODO: Call the correct prepare function
# df, NUMERIC, CATEGORICAL, DROP = prepare_residents()
# df, NUMERIC, CATEGORICAL, DROP = prepare_donors()
# df, NUMERIC, CATEGORICAL, DROP = prepare_social_media()

TARGET = '[TODO: target column name]'

print(f"Dataset shape: {df.shape}")
print(f"Target: '{TARGET}'")
# For classification:
# print(f"Base rate: {df[TARGET].mean():.1%} positive")
# For regression:
# print(f"Target mean: {df[TARGET].mean():.2f}  |  std: {df[TARGET].std():.2f}")

### 2.2 Feature Definition

`define_features()` is called in committed pipeline mode with `DROP['[target]']` —
the per-target drop list returned by `prepare_[domain]()` and defined during EDA.
It prevents:

- **Direct leakage:** [TODO: what column directly encodes the target]
- **Cross-target contamination:** other pipeline targets excluded
- **[TODO: any other leakage specific to this target]**

In [ ]:
X, y = define_features(
    df,
    target=TARGET,
    numeric=NUMERIC,
    categorical=CATEGORICAL,
    drop_cols=DROP[TARGET],
)

# Force categorical columns to string — prevents mixed-type imputer crashes
categorical_in_X = [c for c in CATEGORICAL if c in X.columns]
numeric_in_X     = [c for c in NUMERIC     if c in X.columns]
X[categorical_in_X] = X[categorical_in_X].astype(str).replace({'nan': np.nan, '<NA>': np.nan})

print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")
print(f"  Numeric:     {len(numeric_in_X)}")
print(f"  Categorical: {len(categorical_in_X)}")

### 2.3 Exploratory Confirmation

The EDA for this dataset was conducted in `eda_[domain].ipynb`. The cells below
confirm that the expected signals are present in the prepared feature matrix — they
document what the model has to work with, not drive new decisions.

In [ ]:
# TODO: Top numeric features by correlation / F-score
# For classification:
corr = X[numeric_in_X].corrwith(y).abs().sort_values(ascending=False)
print(f"Top 10 numeric features by |correlation| with {TARGET}:")
print(corr.head(10).round(3).to_string())

# For regression: same code works

In [ ]:
# TODO: Categorical breakdown — pick 1-2 key categorical features
# for col in ['[key_cat_1]', '[key_cat_2]']:
#     if col in X.columns:
#         rate = (pd.concat([X[col], y], axis=1)
#                   .groupby(col)[TARGET]
#                   .agg(['mean', 'count'])
#                   .rename(columns={'mean': 'rate', 'count': 'n'}))
#         print(f"\n{TARGET} rate by {col}:")
#         print(rate.round(3).to_string())

In [ ]:
# TODO: Distribution plots — pick 2-3 key numeric features
# fig, axes = plt.subplots(1, 3, figsize=(14, 4))
# features_to_plot = ['[feat_1]', '[feat_2]', '[feat_3]']
#
# For classification:
# for ax, feat in zip(axes, features_to_plot):
#     if feat not in X.columns: continue
#     for val, label in {0: 'Negative', 1: 'Positive'}.items():
#         ax.hist(X.loc[y == val, feat], alpha=0.6, label=label, bins=20)
#     ax.set_title(feat); ax.set_xlabel('Value'); ax.legend()
# plt.suptitle('Key Feature Distributions by Target', y=1.02)
# plt.tight_layout(); plt.show()
#
# For regression:
# for ax, feat in zip(axes, features_to_plot):
#     ax.scatter(X[feat], y, alpha=0.4, s=15)
#     ax.set_xlabel(feat); ax.set_ylabel(TARGET)
# plt.tight_layout(); plt.show()

---
## 3. Modeling & Feature Selection

### 3.1 Train/Test Split

The test set is locked here and not touched again until Section 4. All modeling
decisions — including hyperparameter tuning — use only training data.

In [ ]:
# TODO: Set stratify=True for classification, False for regression
PROBLEM_TYPE = '[classification / regression]'
stratify     = (PROBLEM_TYPE == 'classification')

X_train, X_test, y_train, y_test = split_data(X, y, stratify=stratify)

### 3.2 Candidate Model Comparison

[TODO: describe the CV setup for this problem type]

For **classification**: 5-fold StratifiedKFold, `class_weight='balanced'` applied
where supported to account for [X]% positive rate.

For **regression**: 5-fold KFold, primary metric is R².

The preprocessor is built unfitted and fit only inside each CV fold — no leakage
from test data into training distributions.

- **Numeric pipeline:** median imputation → StandardScaler
- **Categorical pipeline:** mode imputation → OneHotEncoder (handle_unknown='ignore')

In [ ]:
preprocessor = build_preprocessor(numeric_in_X, categorical_in_X)
pipelines    = build_pipelines(preprocessor, problem_type=PROBLEM_TYPE)

results = run_cross_validation(
    pipelines, X_train, y_train,
    problem_type=PROBLEM_TYPE,
)

### 3.3 Model Selection

**Winner: [TODO: model name]**

<!-- TODO: Justify the choice using the CV output.
Apply the 2x-std rule: if two models are within 2x std of each other on the
primary metric, prefer the simpler one.
Explain what you gain from this choice (interpretability, stability, speed, etc.) -->

### 3.4 Hyperparameter Tuning

In [ ]:
# TODO: Set the winning model name and param grid
WINNER = '[LogisticRegression / RandomForest / GradientBoosting / ...]'

# TODO: Fill in the param grid for the chosen model
# LogisticRegression:
# param_grid = {'model__C': [0.001, 0.01, 0.1, 1.0, 10.0],
#               'model__penalty': ['l1', 'l2'], 'model__solver': ['liblinear']}
#
# RandomForest:
# param_grid = {'model__n_estimators': [100, 200, 300],
#               'model__max_depth': [3, 5, 10, None],
#               'model__min_samples_leaf': [1, 2, 5]}
#
# GradientBoosting:
# param_grid = {'model__n_estimators': [100, 200],
#               'model__learning_rate': [0.01, 0.05, 0.1],
#               'model__max_depth': [3, 5]}

param_grid = {}  # TODO: replace with above

tuned_pipeline, search = tune_model(
    pipeline=pipelines[WINNER],
    X_train=X_train,
    y_train=y_train,
    param_grid=param_grid,
    problem_type=PROBLEM_TYPE,
    search_type='grid',  # or 'random' for larger grids
)

print(f"Best parameters: {search.best_params_}")
print(f"Best CV score:   {search.best_score_:.4f}")

### 3.5 Feature Importance

<!-- TODO: Choose the appropriate importance method for your winning model:

LogisticRegression → extract coef_ (log-odds for classification, direct units for regression)
RandomForest / GradientBoosting → extract feature_importances_
Any model → use permutation importance from fn_model_predict.run_permutation_importance()
-->

In [ ]:
# TODO: Extract and plot feature importance for the winning model

# ── LogisticRegression (classification) ──────────────────────────────────────
# from sklearn.pipeline import Pipeline
# assert isinstance(tuned_pipeline, Pipeline)
# tuned_pipeline.fit(X_train, y_train)
# lr    = tuned_pipeline.named_steps['model']
# prep  = tuned_pipeline.named_steps['preprocessor']
# try:
#     ohe_names = (prep.named_transformers_['cat']
#                      .named_steps['onehot']
#                      .get_feature_names_out(categorical_in_X).tolist())
# except Exception:
#     ohe_names = []
# coef_series = pd.Series(lr.coef_[0], index=numeric_in_X + ohe_names)
# print(coef_series.abs().nlargest(15).round(3).to_string())

# ── Tree-based models ─────────────────────────────────────────────────────────
# tuned_pipeline.fit(X_train, y_train)
# model = tuned_pipeline.named_steps['model']
# prep  = tuned_pipeline.named_steps['preprocessor']
# try:
#     ohe_names = (prep.named_transformers_['cat']
#                      .named_steps['onehot']
#                      .get_feature_names_out(categorical_in_X).tolist())
# except Exception:
#     ohe_names = []
# importances = pd.Series(model.feature_importances_, index=numeric_in_X + ohe_names)
# importances.nlargest(15).sort_values().plot(kind='barh', figsize=(10, 6))
# plt.title('Feature Importances'); plt.tight_layout(); plt.show()

pass  # remove this when you fill in the block above

---
## 4. Evaluation & Interpretation

### 4.1 Final Test Set Evaluation

The test set was locked in Section 3.1. This is its one use — evaluating the tuned
pipeline on data it has never seen.

In [ ]:
metrics, final_pipeline = evaluate_final_model(
    tuned_pipeline, X_train, y_train, X_test, y_test,
    problem_type=PROBLEM_TYPE,
)

### 4.2 [Threshold Analysis (classification) / Residual Analysis (regression)]

<!-- TODO: Choose the right diagnostic for your problem type.

CLASSIFICATION — threshold analysis:
Describe the false positive / false negative cost asymmetry for this specific
problem. Which error is worse and why? What threshold maximizes the right metric?

REGRESSION — residual analysis:
Plot residuals vs. predicted values. Check for heteroscedasticity, outliers,
and systematic bias. Compare RMSE to the baseline (predict-mean).
-->

In [ ]:
# TODO: Diagnostic plots

# ── Classification: PR and ROC curves ────────────────────────────────────────
# from sklearn.metrics import precision_recall_curve, roc_curve, auc
# y_proba = final_pipeline.predict_proba(X_test)[:, 1]
# prec, rec, pr_t = precision_recall_curve(y_test, y_proba)
# fpr, tpr, _    = roc_curve(y_test, y_proba)
# fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# axes[0].plot(rec, prec, color='steelblue', lw=2)
# axes[0].axhline(y_test.mean(), color='gray', linestyle='--', label='No-skill')
# axes[0].set(xlabel='Recall', ylabel='Precision',
#             title=f'PR Curve (AUC={auc(rec,prec):.3f})')
# axes[0].legend()
# axes[1].plot(fpr, tpr, color='steelblue', lw=2); axes[1].plot([0,1],[0,1],'k--')
# axes[1].set(xlabel='FPR', ylabel='TPR', title=f'ROC (AUC={auc(fpr,tpr):.3f})')
# plt.tight_layout(); plt.show()

# ── Regression: residuals ─────────────────────────────────────────────────────
# y_pred = final_pipeline.predict(X_test)
# resid  = y_test - y_pred
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].scatter(y_pred, resid, alpha=0.5, s=20)
# axes[0].axhline(0, color='red', linestyle='--')
# axes[0].set(xlabel='Predicted', ylabel='Residual', title='Residuals vs Predicted')
# axes[1].hist(resid, bins=25, edgecolor='white')
# axes[1].set(xlabel='Residual', title='Residual Distribution')
# plt.tight_layout(); plt.show()

pass  # remove when filled in

### 4.3 Business Interpretation

<!-- TODO: Interpret the results in plain language for a non-technical stakeholder.
Do not just restate the metric numbers — explain what they mean for decisions.

For classification:
  - What does this ROC-AUC / F1 mean operationally?
  - What is the recommended threshold and why?
  - What are the consequences of the remaining errors?

For regression:
  - How much better is the model than a baseline?
  - What does the RMSE mean in the units of the target?
  - Where is the model weakest?
-->

---
## 5. Causal and Relationship Analysis

### 5.1 [Explanatory Model Setup]

<!-- TODO: Describe what explanatory model you are fitting and why.

For classification targets: fit_causal_classification() → statsmodels Logit
For regression targets:     fit_causal_regression()     → statsmodels OLS

Note: if p > n after one-hot encoding (more columns than rows), use SelectKBest
to reduce to a manageable feature set before fitting. This is a known limitation
with wide feature matrices and small samples. Document what you selected and why.

Always include the disclaimer: significant coefficients are associations, not
causal effects. Name specific plausible confounders for this domain.
-->

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, f_regression

# One-hot encode for statsmodels
X_train_enc = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_train_enc = X_train_enc.apply(pd.to_numeric, errors='coerce').fillna(0)

n_rows, n_cols = X_train_enc.shape
print(f"Encoded matrix: {n_rows} rows × {n_cols} columns")

if n_cols >= n_rows:
    # p > n: reduce with SelectKBest before statsmodels
    k = min(10, n_rows - 5)
    # TODO: use f_classif for classification, f_regression for regression
    selector = SelectKBest(score_func=f_classif, k=k)
    selector.fit(X_train_enc, y_train.astype(int))
    top_cols = X_train_enc.columns[selector.get_support()]
    X_causal = X_train_enc[top_cols]
    print(f"Reduced to {k} features for causal model: {list(top_cols)}")
else:
    X_causal = X_train_enc
    print("Matrix is safe for statsmodels without reduction.")

In [ ]:
# TODO: Call the correct causal fitting function
# from functions.fn_model_causal import fit_causal_classification, get_coefficients
# causal_results = fit_causal_classification(X_causal, y_train.astype(int))

# from functions.fn_model_causal import fit_causal_regression, get_coefficients
# causal_results = fit_causal_regression(X_causal, y_train)

# print(causal_results.summary())

In [ ]:
# TODO: Extract significant coefficients
# coef_df = get_coefficients(causal_results, model_type='[logistic / linear]')
# sig = coef_df[coef_df['p_value'] < 0.05].sort_values('coefficient', ascending=False)
# print("Significant features (p < 0.05):")
# print(sig[['feature', 'coefficient', 'p_value', 'significant']].to_string(index=False))

pass

### 5.2 Relationship Interpretation

<!-- TODO: Interpret the significant coefficients in domain terms.

Structure your interpretation:
1. Group significant features into 2-3 thematic clusters
2. For each cluster, explain what the association suggests and whether it
   aligns with domain knowledge or theory
3. Name at least one specific confounder that prevents causal interpretation
4. State one actionable (correlation-based) insight for the organization

Always close with the explicit reminder: this is a correlation, not causation.
-->

---
## 6. Deployment

The trained pipeline is saved as a `.pkl` file. The inference endpoint, .NET controller
integration, React dashboard component, and retraining schedule are documented
separately in `ml-pipelines/deployment-notes.md`.

In [ ]:
os.makedirs('models', exist_ok=True)

pkl_path = save_model(
    final_pipeline,
    metrics,
    target_name=TARGET,
    output_dir='models',
)

print(f"Model saved: {pkl_path}")

---
*Hearth Haven — IS 455 INTEX Pipeline*